# Word2Vec With Pytorch


```
Raw text
  ↓
Tokenization
  ↓
Vocabulary
  ↓
Center-context pairs
  ↓
Embedding lookup
  ↓
Dot product similarity
  ↓
Negative sampling loss
  ↓
Update vectors
  ↓
Useful word embeddings
```

In [1]:
import torch
from staticvectors import StaticVectors


c:\Users\108pa\miniconda3\envs\py3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Using StaticVectors Library

In [2]:
model = StaticVectors("neuml/word2vec")

In [ ]:
sentence = "it's a jolly good day mate"
words = sentence.split()

# Get the vectors for each word
vectors = model.embeddings(words)

print(vectors)

## Using Tensorflow

In [3]:
!pip install tensorflow-cpu

In [4]:
import io
import re
import string
import tensorflow as tf
from tensorflow.keras import layers

In [5]:

SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# 1. Download Shakespeare corpus
path_to_file = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path_to_file, "r", encoding="utf-8").read()


1115394/1115394 [==============================] - 0s 0us/step


In [6]:
### Clean up and tokenize the corpus 

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned = clean_text(text)
tokens = cleaned.split()

print("Total tokens:", len(tokens))
print(tokens[:30])

Total tokens: 202619
['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen']


In [7]:
### Build vocabulary

vocab_size = 10000

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens = vocab_size,
    standardize  = None,
    split = "whitespace",
    output_mode = "int"
)

text_ds = tf.data.Dataset.from_tensor_slices([cleaned]).batch(1)
vectorize_layer.adapt(text_ds)

vocab = vectorize_layer.get_vocabulary()

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}

print("Vocab size: ", len(vocab))
print(vocab[:30])

Vocab size:  10000
['', '[UNK]', 'the', 'and', 'to', 'i', 'of', 'you', 'my', 'a', 'that', 'in', 'is', 'not', 'for', 'with', 'me', 'it', 'be', 'your', 'his', 'this', 'but', 'he', 'have', 'as', 'thou', 'him', 'so', 'what']


In [8]:
### Convert full corpus to integer IDs

sequence = vectorize_layer(tf.constant([cleaned]))[0].numpy()

# Remove zeros if any padding exists 
sequence = sequence[sequence != 0]

print(sequence[:30])
print([id_to_word[i] for i in sequence[:30]])

[  89  270  138   36  981  144  673  125   16  106   34  106  106   89
  270    7   41   34 1285  344    4  200   64    4 3689   34 1285 1285
   89  270]
['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen']


In [9]:
### Generate Skip-gram pairs

window_size = 2

positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
    sequence,
    vocabulary_size = len(vocab),
    window_size = window_size,
    negative_samples = 0
)

print("Positive pairs: ", len(positive_skip_grams))

for target, context in positive_skip_grams[:10]:
    print(id_to_word[target], "->", id_to_word[context])

Positive pairs:  810470
shall -> [UNK]
no -> measure
said -> pity
was -> to
off -> blood
stand -> joy
hast -> poison
to -> touch
is -> so
single -> sole


In [ ]:
### Add negative samples 

num_ns = 4
targets = []
contexts = []
labels = []

sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(len(vocab))

for target_word, context_word in positive_skip_grams:
    context_class = tf.reshape(tf.constant(context_word, dtype=tf.int64),(1,1))

    negative_samples, _, _ = tf.random.log_uniform_candidate_sampler(
        true_classes=context_class,
        num_true=1,
        num_sampled=num_ns,
        unique=True,
        range_max=len(vocab),
        seed=SEED
    )

    context = tf.concat(
        [tf.constant([context_word], dtype=tf.int64), negative_samples],
        axis = 0
    )

    label = tf.constant([1] + [0] * num_ns, dtype=tf.int64)

    targets.append(target_word)
    contexts.append(context.numpy())
    labels.append(label.numpy())


targets = np.array(targets)
contexts = np.array(contexts)
labels = np.array(labels)

print(targets.shape)
print(contexts.shape)
print(labels.shape)

In [ ]:
### Build Dataset 

BATCH_SIZE = 1024
BUFFER_SIZE = 10000

dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.cache().prefetch(AUTOTUNE)

In [ ]:
### Word2Vec Model

class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, num_ns):
        super().__init__()
        self.target_embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, name="target_embedding")
        self.context_embedding= tf.keras.layers.Embedding(vocab_size, embedding_dim, name="context_embedding")


    def call(self, pair):
        target, context = pair
        word_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)

        dots = tf.einsum("be,bce->bc", word_emb, context_emb)

        return dots

In [ ]:
!pip install tensorflow-directml-plugin


In [ ]:
### Training model

embedding_dim = 128

word2vec = Word2Vec(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    num_ns=num_ns
)

word2vec.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history = word2vec.fit(dataset, epochs=10)